# Notebook 03: Portfolio Analysis and Risk Metrics
This notebook demonstrates portfolio construction, efficient frontier generation, and risk metric reporting for selected funds.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize

ROOT_DIR = Path.cwd()
RAW_DIR = ROOT_DIR / "data" / "raw"

nav_history = pd.read_csv(RAW_DIR / "02_nav_history.csv")
fund_master = pd.read_csv(RAW_DIR / "01_fund_master.csv")

nav_history["date"] = pd.to_datetime(nav_history["date"], errors="coerce")
nav_history = nav_history.dropna(subset=["date", "nav", "amfi_code"])

fund_master["scheme_name"] = fund_master.get("scheme_name", fund_master.get("fund_name", pd.Series(["Unknown"] * len(fund_master))))
merged = nav_history.merge(fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left")
merged["fund_name"] = merged["scheme_name"].fillna("Unknown")

selected = [
    "HDFC Top 100",
    "SBI Bluechip",
    "ICICI Bluechip",
    "Nippon Large Cap",
    "Axis Bluechip",
]

returns = (
    merged[merged["fund_name"].isin(selected)]
    .sort_values(["fund_name", "date"])
    .pivot(index="date", columns="fund_name", values="nav")
    .pct_change()
    .dropna()
)
returns.head()

In [ ]:
# Compute the mean returns and covariance matrix for the selected funds
mean_returns = returns.mean()
cov_matrix = returns.cov()
num_assets = len(returns.columns)

print("Selected funds:", returns.columns.tolist())
print("Mean daily returns:\n", mean_returns)
print("Covariance matrix:\n", cov_matrix)

In [ ]:
# Optimize for the efficient frontier

def portfolio_volatility(weights, cov):
    return np.sqrt(weights.T @ cov @ weights)


def portfolio_return(weights, mean_ret):
    return weights @ mean_ret


bounds = tuple((0.0, 1.0) for _ in range(num_assets))
constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)

returns_range = np.linspace(mean_returns.min(), mean_returns.max(), 20)
frontier = []
for target in returns_range:
    cons = (
        constraints,
        {"type": "eq", "fun": lambda w, target=target: portfolio_return(w, mean_returns) - target},
    )
    result = minimize(
        lambda w: portfolio_volatility(w, cov_matrix),
        np.repeat(1.0 / num_assets, num_assets),
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
    )
    if result.success:
        frontier.append({
            "target_return": target,
            "volatility": portfolio_volatility(result.x, cov_matrix),
            **{name: weight for name, weight in zip(returns.columns, result.x)},
        })

frontier_df = pd.DataFrame(frontier)
frontier_df.head()

In [ ]:
# Find the maximum Sharpe ratio portfolio based on annualized return/volatility
risk_free_rate = 0.065

def negative_sharpe(weights):
    ret = portfolio_return(weights, mean_returns) * 252
    vol = portfolio_volatility(weights, cov_matrix) * np.sqrt(252)
    return -((ret - risk_free_rate) / vol)

result = minimize(
    negative_sharpe,
    np.repeat(1.0 / num_assets, num_assets),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

best_weights = pd.Series(result.x, index=returns.columns)
print("Max Sharpe weights:")
print(best_weights)